# Transcripción del conjunto externo

Tres obras con edición Braille compás sobre compás hecha por personas (biblioteca BrailleOrch) y grabación de dominio público o CC0 en Internet Archive. No se usan para elegir parámetros; sirven para comprobar si lo que se ajusta con las Goldberg se sostiene en obras de otro estilo, otro intérprete y otro transcriptor Braille.

| Obra | Braille | Grabación |
| --- | --- | --- |
| Beethoven, Für Elise | BrailleOrch Bor001 | `FurElise_201603` |
| Debussy, La fille aux cheveux de lin | BrailleOrch Bor195 | `gyr-la-fille-aux-cheveux-de-lin-debussy` |
| Chopin, Vals en la menor B. 150 | BrailleOrch Bor343 | `parso-08e57b88c8badbff300ef033655023e035bbfe2a` |

El notebook solo guarda las notas, igual que `amt-goldberg-transcribe`; la traducción y el BSA se calculan fuera con el código del repositorio.

## 1. Instalación

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("--no-deps", "piano_transcription_inference", "torchlibrosa", "mido")
pip("librosa", "soundfile")

## 2. Grabaciones

In [ ]:
import json, time, urllib.parse, urllib.request
from pathlib import Path

AUDIO = Path("/kaggle/working/audio")
AUDIO.mkdir(parents=True, exist_ok=True)
OBRAS = {
    "fur_elise": ("FurElise_201603", "Fur Elise.mp3"),
    "fille": ("gyr-la-fille-aux-cheveux-de-lin-debussy", "Gyr - La Fille aux cheveux de Lin (Debussy).mp3"),
    "vals_chopin": ("parso-08e57b88c8badbff300ef033655023e035bbfe2a", "000_Waltz_no_19_-_op_posth_A_minor.mp3"),
}
for nombre, (item, archivo) in OBRAS.items():
    destino = AUDIO / f"{nombre}.mp3"
    if not destino.exists():
        urllib.request.urlretrieve(f"https://archive.org/download/{item}/" + urllib.parse.quote(archivo), destino)
    print(nombre, destino.stat().st_size // 1024, "KB")

## 3. Modelos y normalización

El normalizador es el mismo de `src/amt/audio.py`: quita la componente continua y lleva el RMS de los tramos con sonido a -24.9 dBFS sin recortar picos.

In [ ]:
import glob
import numpy as np
import librosa
import torch
from piano_transcription_inference import PianoTranscription, sample_rate as SR

TARGET_DBFS = -24.9

def db(x):
    return 20.0 * np.log10(max(x, 1e-12))

def active_rms(x, frame=2048):
    n = len(x) // frame
    rms = np.sqrt(np.mean(x[: n * frame].reshape(n, frame) ** 2, axis=1))
    loud = rms[rms >= rms.max() * 10 ** (-40 / 20)]
    return float(np.sqrt(np.mean(loud ** 2)))

def normalize_loudness(x):
    x = np.asarray(x, dtype=np.float32) - np.mean(x)
    level = active_rms(x)
    if db(level) < -80:
        return x
    gain = 10 ** ((TARGET_DBFS - db(level)) / 20)
    gain = min(gain, 0.99 / (np.abs(x).max() + 1e-12))
    return (x * gain).astype(np.float32)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE = str(Path.home() / "piano_transcription_inference_data" / "note_F1=0.9677_pedal_F1=0.9186.pth")
TUNED = glob.glob("/kaggle/input/**/kong_ajustado.pth", recursive=True)
assert TUNED, "Agregar la salida de amt-finetune como input del notebook"
MODELS = {"original": BASE, "ajustado": TUNED[0]}
print(DEVICE, MODELS)

## 4. Transcripción

In [ ]:
OUT = Path("/kaggle/working/externo")
OUT.mkdir(exist_ok=True)

for model_name, ckpt in MODELS.items():
    transcriber = PianoTranscription(device=DEVICE, checkpoint_path=ckpt)
    for nombre in OBRAS:
        destino = OUT / f"{model_name}__{nombre}.json"
        if destino.exists():
            continue
        audio, _ = librosa.load(str(AUDIO / f"{nombre}.mp3"), sr=SR, mono=True)
        audio = normalize_loudness(audio)
        t0 = time.time()
        out = transcriber.transcribe(audio, None)
        latency = time.time() - t0
        notes = [[float(e["onset_time"]), float(e["offset_time"]), int(e["midi_note"]), int(e["velocity"])]
                 for e in out["est_note_events"]]
        pedals = [[float(e["onset_time"]), float(e["offset_time"])] for e in out.get("est_pedal_events", [])]
        destino.write_text(json.dumps({
            "pieza": nombre, "archivo": OBRAS[nombre][1], "modelo": model_name,
            "duracion_s": len(audio) / SR, "latencia_s": latency,
            "notas": notes, "pedal": pedals,
        }))
        print(f"{model_name:9s} {nombre:12s} {len(notes):5d} notas  {latency:5.1f} s")
    del transcriber
    torch.cuda.empty_cache()